# 过滤、平滑与在线变点

先计算 Bayes 权重，再观察一个在线警报的误报与延迟。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(20260907)
np.set_printoptions(precision=5, suppress=True)

有限状态更新使用先验乘似然，再归一化。

In [ ]:
prior=np.array([.8,.2]);likelihood=np.array([.1,.6]);unnormalized=prior*likelihood
posterior=unnormalized/unnormalized.sum();P=np.array([[.9,.1],[.2,.8]])
print('weights, posterior, next prior:',unnormalized,posterior,posterior@P)

局部水平滤波逐步显示增益，保存后验用于事后平滑比较。

In [ ]:
Q=.1;R=.2;m=0.;v=1.;observations=np.array([0,.2,2.8,3.1]);means=[];variances=[]
for y in observations:
    vp=v+Q;gain=vp/(vp+R);m=m+gain*(y-m);v=(1-gain)*vp
    means.append(m);variances.append(v)
smooth=np.array(means)
for t in range(len(means)-2,-1,-1):
    J=variances[t]/(variances[t]+Q);smooth[t]=means[t]+J*(smooth[t+1]-means[t])
print('filter:',means,'smoother:',smooth)
plt.plot(observations,'o',label='observed');plt.plot(means,label='filtered');plt.plot(smooth,label='smoothed');plt.legend();plt.show()

模拟无变化与均值上移两类路径。阈值改变误报和延迟，而非只改变一条图。

In [ ]:
threshold=5.;allowance=.5;T=100;change=50;repeats=500
for shifted in [False,True]:
    alarms=[]
    for repeat in range(repeats):
        residual=rng.normal(size=T)
        if shifted:residual[change:]+=1
        score=0.;alarm=T
        for t,e in enumerate(residual):
            score=max(0,score+e-allowance)
            if score>threshold:alarm=t;break
        alarms.append(alarm)
    alarms=np.array(alarms)
    detected=(alarms>=change)&(alarms<T)
    print('shifted:',shifted,'alarm fraction:',(alarms<T).mean(),'early fraction:',(alarms<change).mean(),'mean delay if detected:',np.mean(alarms[detected]-change) if detected.any() else None)

## 自己试一试

为什么第二期平滑状态不适合用于第二期交易？

## 反馈

后向递推读取了第三、四期观测；它是事后估计，不属于第二期信息。

参数改变后应重新解释结果，不要求复现某次随机实验的小数。